# IS2083 Lab 5: AI Automation Tool, Part 1

**Before you begin, you need TWO things:**

1. Your **Jupyter login** (you used it to open this notebook).
2. Your **API key** for the class Qwen Coder model server (from your instructor).

This lab assumes you already know basic Python. The new material is the **AI model**, its **API**, and **prompt design**. You will connect to the class **Qwen Coder** model, use it as a coding assistant, and build an **AI-powered security log assessment** tool. At the end you will feed it a real security log exported from **Windows** or **macOS** and have it write an assessment report. Run cells with **Shift + Enter**, top to bottom.

## Step 1: Install the openai client (run once)

In [1]:
%pip install openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Step 2: Connect to the Qwen Coder model server

Paste the `BASE_URL` (ends in `/v1`) and your `API_KEY` from your instructor. The code then queries the server for the served model id.

In [2]:
from openai import OpenAI

BASE_URL = "http://gpustack.cds.utsa.edu/v1"   # class model server address
API_KEY  = "PASTE_YOUR_API_KEY_HERE"  # your personal API key (remove before GitHub!)

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
MODEL = client.models.list().data[1].id   # discover the served model id
print("Connected. Model:", MODEL)

Connected. Model: qwen3-vl-8b-instruct


## Step 3: A reusable model helper (with retries)

`ask_model` wraps one chat completion and retries on transient errors.

In [3]:
import time

def ask_model(prompt: str, temperature: float = 0.2, retries: int = 2) -> str:
    """Send one prompt to Qwen Coder and return its text reply, retrying on failure."""
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
            )
            return resp.choices[0].message.content
        except Exception:
            if attempt == retries:
                raise
            time.sleep(1.5 * (attempt + 1))

print(ask_model("Say hello in one short sentence."))

Hello!


## Step 4: Plan the tool

Input: one security log entry. Output: a structured result with a `label` (BENIGN / SUSPICIOUS / MALICIOUS) and a one-sentence `reason`. You solve the structure problem with a strict output format in the prompt plus defensive parsing.

## Step 5: Use Qwen Coder as a coding assistant

Ask the model to sketch an approach, then keep only what you understand.

In [4]:
question = (
    "I am writing a Python function that asks an LLM to classify a security log entry "
    "and returns a dataclass with label and reason. Suggest a robust prompt format and "
    "a defensive way to parse the reply. Keep it concise."
)
print(ask_model(question))

```python
from dataclasses import dataclass
from typing import Optional

@dataclass
class SecurityLogClassification:
    label: str
    reason: str

def classify_log(llm, log_entry: str) -> SecurityLogClassification:
    prompt = f"""
Classify the following security log entry into one of: [NORMAL, SUSPICIOUS, ATTACK].

Log: {log_entry}

Respond ONLY with:
LABEL: <label>
REASON: <brief justification>

Example:
LABEL: NORMAL
REASON: No anomalous activity detected.
    """
    response = llm(prompt)
    lines = response.strip().splitlines()
    label = reason = ""
    for line in lines:
        if line.startswith("LABEL:"):
            label = line.split(":", 1)[1].strip()
        elif line.startswith("REASON:"):
            reason = line.split(":", 1)[1].strip()
    if not label or not reason:
        raise ValueError("LLM response missing required fields")
    return SecurityLogClassification(label, reason)
```


## Step 6: Define the result type and parsers

An `Enum` fixes the labels, a `dataclass` gives a typed result, and two regexes extract the fields.

In [5]:
from dataclasses import dataclass
from enum import Enum
import re

class Label(str, Enum):
    BENIGN = "BENIGN"
    SUSPICIOUS = "SUSPICIOUS"
    MALICIOUS = "MALICIOUS"
    UNKNOWN = "UNKNOWN"

@dataclass
class Triage:
    line: str
    label: Label
    reason: str

LABEL_RE = re.compile(r"^LABEL:\s*(\w+)", re.IGNORECASE | re.MULTILINE)
REASON_RE = re.compile(r"^REASON:\s*(.+)", re.IGNORECASE | re.MULTILINE)

## Step 7: Name your tool

This tool can go in your professional portfolio, so give it a name you would be proud to show. `triage_log` is a fine name for the function, but the **tool** itself can be called anything: Play it straight (SentinelScan, ThreatLens, LogHawk, WatchTower) or give it some personality (SnitchBot, LogFather, GrepReaper, Sherlock Logs, The Log Whisperer, LogNessMonster), your call. Set it here; it appears in your report and your repository name.

In [6]:
TOOL_NAME = "EventGuard"   # <-- change this to your own creative name

## Step 8: Build the tool

Write the classifier: a strict prompt, one `ask_model` call, then defensive parsing that falls back to `Label.UNKNOWN`. (You may rename `triage_log` to match your tool if you like.)

In [8]:
def triage_log(line: str) -> Triage:
    # TODO 1: write a prompt that demands EXACTLY these two lines back:
    #   LABEL: <BENIGN|SUSPICIOUS|MALICIOUS>
    #   REASON: <one sentence>
    prompt = (
        "You are a security analyst. Classify the log entry below as exactly one of: "
        "BENIGN, SUSPICIOUS, or MALICIOUS.\n"
        "Reply with EXACTLY these two lines and nothing else:\n"
        "LABEL: <BENIGN|SUSPICIOUS|MALICIOUS>\n"
        "REASON: <one sentence>\n\n"
        f"Log entry:\n{line}\n"
    )

    text = ask_model(prompt)

    # TODO 2: parse with LABEL_RE and REASON_RE; fall back to Label.UNKNOWN
    label = Label.UNKNOWN
    reason = ""

    m = LABEL_RE.search(text or "")
    if m and m.group(1).upper() in Label.__members__:
        label = Label[m.group(1).upper()]

    r = REASON_RE.search(text or "")
    if r:
        reason = r.group(1).strip()

    return Triage(line=line, label=label, reason=reason)

## Step 9: Quick sanity check
Confirm the tool works on one entry before feeding it a file.

In [9]:
print(triage_log("Failed password for root from 203.0.113.9 port 4022 ssh2"))

Triage(line='Failed password for root from 203.0.113.9 port 4022 ssh2', label=<Label.SUSPICIOUS: 'SUSPICIOUS'>, reason='Multiple failed login attempts for root user indicate potential brute force attack.')


## Step 10: Assess a real security log file

Export your computer's security log to **one file** and upload it here. Do the ONE that matches your computer; you only turn in one (Windows or macOS).

**Windows (Event Viewer):** open Event Viewer, expand **Windows Logs**, click **Security**. In the right panel click **Save All Events As...**, set *Save as type* to **CSV (Comma Separated)**, and save it as `security_log.csv`.

**macOS (Terminal):** export recent security-related events to one file:

```
log show --predicate 'process == "sudo" OR process == "sshd" OR process == "loginwindow" OR eventMessage CONTAINS[c] "authentication"' --last 6h --style syslog > security_log.txt
```

If that file is nearly empty, use: `log show --last 1h --style syslog > security_log.txt`

Then click **Upload** in the Jupyter file browser and add your `security_log.csv` (Windows) or `security_log.txt` (macOS) to this notebook's folder. Set `LOG_FILE` below to match.

In [10]:
LOG_FILE = "security_log.csv"   # Windows CSV, or "security_log.txt" on macOS
MAX_ENTRIES = 20                 # keep this first version small and quick

def read_log_lines(path, limit=MAX_ENTRIES):
    with open(path, encoding="utf-8", errors="ignore") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    return lines[:limit]

entries = read_log_lines(LOG_FILE)
results = [triage_log(line) for line in entries]
print(f"Assessed {len(results)} entries from {LOG_FILE}")

Assessed 20 entries from security_log.csv


## Step 11: Write the assessment report

Have your tool write a Markdown report named after your tool. This report is your deliverable.

In [11]:
from datetime import datetime

def write_report(results, tool_name, path="assessment.md"):
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"# {tool_name} Security Log Assessment\n\n")
        f.write(f"Generated: {datetime.now():%Y-%m-%d %H:%M}\n\n")
        f.write(f"Entries assessed: {len(results)}\n\n")
        for r in results:
            f.write(f"- **{r.label.value}** {r.reason}\n")
            f.write(f"  - `{r.line}`\n")
    return path

report_path = write_report(results, TOOL_NAME)
print("Wrote", report_path, "\n")
print(open(report_path, encoding="utf-8").read())

Wrote assessment.md 

# EventGuard Security Log Assessment

Generated: 2026-11-16 14:44

Entries assessed: 20

- **BENIGN** The log entry appears to be a header row for a log table, not an actual event or activity.
  - `﻿Keywords,Date and Time,Source,Event ID,Task Category`
- **BENIGN** The log entry indicates a successful special logon with assigned privileges, which is a normal audit event for system administration activities.
  - `Audit Success,11/16/2026 2:32:54 PM,Microsoft-Windows-Security-Auditing,4672,Special Logon,"Special privileges assigned to new logon.`
- **BENIGN** The log entry contains no actionable data or indicators of compromise.
  - `Subject:`
- **BENIGN** SYSTEM is a legitimate Windows account with no indication of malicious activity.
  - `Security ID:		SYSTEM`
- **BENIGN** SYSTEM account is a legitimate built-in account used for system processes and is not inherently malicious.
  - `Account Name:		SYSTEM`
- **BENIGN** NT AUTHORITY is a built-in Windows system acco

## Step 12: Read, test, and trust

Double-click to edit and answer briefly:

1. Why model the output with an `Enum` and `dataclass` instead of a bare dict?
2. What does your defensive parsing do when the model ignores the format?
3. Which entries in your report look wrong, and how would you catch that?

_Your answers:_
Q1:The Label Enum allows only four answers: BENIGN, SUSPICIOUS, MALICIOUS, or UNKNOWN. If the AI makes up a different word, my code catches it and uses UNKNOWN instead. The Triage dataclass makes every result have the same three parts (line, label, and reason), which a plain dict doesn’t promise.
Q2:Nothing crashes, and nothing gets skipped. The label starts as UNKNOWN and only changes if the AI gives a real label, so a messy answer stays UNKNOWN with an empty reason (unless a REASON line was found). The entry still shows up in the report.
Q3:The header row got sent to the AI and labeled like it was a real event. One event was also split into many small entries, like Subject: and Security ID:, because my code splits the log at every new line. That’s like grading strips torn from one page. To catch this, I could skip the first line, flag entries that are just a label ending in a colon, and check that the number of results matches the number of real events.

## Step 13: Save and submit

1. **File > Save Notebook**, then **File > Download** for the `.ipynb`.
2. Download your report `assessment.md` too.
3. On GitHub, create a **public** repository **named after your tool** (for example `sentinelscan`). Do **not** name it lab5 or use the course name.
4. **Add file > Upload files**: upload your notebook and `assessment.md`. Do **not** upload the raw security log; it may contain private data.
5. Before uploading, **remove your real API key** from the connect cell.
6. Take the screenshots listed in the handout into the submission template.